<a href="https://colab.research.google.com/github/saeid-uot/3253-Machine-Learning/blob/main/Week_08%20-%20Dimensionality%20Reduction/1-%20Solution%20PCA%20MNIST%20Student%20Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective:

### Use MNIST dataset and apply PCA to find out the impact on the model training time and also model performance
### The work is taken from https://github.com/mGalarnyk/Python_Tutorials/blob/master/Sklearn/PCA/PCA_to_Speed-up_Machine_Learning_Algorithms.ipynb

In [1]:
# Setup
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml


#Download and Load the Data


In [2]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, cache=True)
mnist.target = mnist.target.astype(np.int8) # fetch_openml() returns targets as strings

X, y = mnist["data"], mnist["target"]

# Split data into train/test

In [3]:
# Write a code to split your dataset into 80/20 dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size= 0.2)

# View Data Dimension

In [4]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((56000, 784), (14000, 784), (56000,), (14000,))

#Standardizing the Data¶

Since PCA yields a feature subspace that maximizes the variance along the axes, it makes sense to standardize the data, especially, if it was measured on different scales.

Standardization of a dataset is a common requirement for many machine learning estimators: they might behave badly if the individual feature do not more or less look like standard normally distributed data

Notebook going over the importance of feature Scaling: http://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html#sphx-glr-auto-examples-preprocessing-plot-scaling-importance-py


In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit on training set only.
scaler.fit(X_train)

# Apply transform to both the training set and the test set.
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

#X_train.shape, X_test.shape, y_train.shape, y_test.shape


# Lets fit a simple model

In [6]:
from sklearn.linear_model import LogisticRegression
logisticRegr = LogisticRegression(multi_class ='multinomial', max_iter=1000)

import datetime
start= datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end= datetime.datetime.now()

#time taken to train the model
print(end-start)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


0:01:06.781723


#Measure the accuracy of the model before PCA

In [7]:
from sklearn.metrics import accuracy_score

acc_train = accuracy_score(y_train, logisticRegr.predict(X_train))
acc_test = accuracy_score(y_test, logisticRegr.predict(X_test))

print("Performance on train:" , acc_train ,"\n", "Performance on test:", acc_test)

Performance on train: 0.9440714285714286 
 Performance on test: 0.9161428571428571


In [8]:
# In case you want to see how the scaled number would look like, you can uncomment below lines
#from scipy.stats import describe
#describe(X_train)[1]

#Now, lets implement PCA

In [9]:
from sklearn.decomposition import PCA
# specify how much of variation you would like PCA to capture (between 0-1)
# pca = PCA('mle')
pca = PCA(0.5)

pca.fit(X_train)


PCA(n_components=0.5)

# Look at components

In [10]:
pca.n_components_


38

#Apply the mapping (transform) to both the training set and the test set.



In [11]:
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)

#Build a linear model and measure model fitting period.

In [12]:
from sklearn.linear_model import LogisticRegression
logisticRegr = LogisticRegression(multi_class ='auto')

import datetime
start= datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end= datetime.datetime.now()

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#measure training time

In [13]:

print(end-start)
#logisticRegr.predict(X_train[0].reshape(1,-1))


0:00:03.586552


#Measuring Model Performance

In [14]:
score = logisticRegr.score(X_test, y_test)
print(score)

0.8985714285714286


#Now lets build a loop to go through various value of PCA n_component, measure training time and accuracy

In [15]:
import pandas as pd

pca_values = [0.99, 0.98, 0.95, 0.90, 0.95, 0.6]
results = []

for pca_val in pca_values:
    pca = PCA(pca_val)
    pca.fit(X_train)
    X_train_pca = pca.transform(X_train)
    X_test_pca = pca.transform(X_test)

    logisticRegr = LogisticRegression(multi_class='multinomial', max_iter=1000) # Increased max_iter

    start_time = datetime.datetime.now()
    logisticRegr.fit(X_train_pca, y_train)
    end_time = datetime.datetime.now()

    training_time = end_time - start_time

    train_accuracy = accuracy_score(y_train, logisticRegr.predict(X_train_pca))
    test_accuracy = accuracy_score(y_test, logisticRegr.predict(X_test_pca))

    results.append([pca_val, round(training_time.total_seconds(), 3), train_accuracy, test_accuracy])


df = pd.DataFrame(results, columns=['PCA Value', 'Training Time', 'Train Accuracy', 'Test Accuracy'])
df


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

,PCA Value,Training Time,Train Accuracy,Test Accuracy
0,0.99,5.457,0.899500,0.898571
1,0.98,7.422,0.899036,0.896929
2,0.95,5.274,0.895250,0.893286
3,0.90,6.465,0.891446,0.889429
4,0.95,5.234,0.895250,0.893286
5,0.60,4.182,0.819946,0.815143


In [ ]:
# prompt: repeat the above code on a larger training set bigger than mnist to show the impact of PCA.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
import datetime
from sklearn.metrics import accuracy_score

# Use a larger dataset than MNIST
# Example: Fashion-MNIST
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, cache=True)
fashion_mnist.target = fashion_mnist.target.astype(np.int8)

X, y = fashion_mnist["data"], fashion_mnist["target"]

# Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Added random_state for reproducibility

# Standardizing the Data
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Train a Logistic Regression model without PCA
logisticRegr = LogisticRegression(multi_class='multinomial', max_iter=1000) # Increased max_iter for convergence

start = datetime.datetime.now()
logisticRegr.fit(X_train, y_train)
end = datetime.datetime.now()

print("Training time without PCA:", end - start)

acc_train = accuracy_score(y_train, logisticRegr.predict(X_train))
acc_test = accuracy_score(y_test, logisticRegr.predict(X_test))

print("Performance without PCA:\nTrain accuracy:", acc_train, "\nTest accuracy:", acc_test)


# Apply PCA
pca = PCA(0.95) # Example: retain 95% of variance
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

# Train a Logistic Regression model with PCA
logisticRegr_pca = LogisticRegression(multi_class='multinomial', max_iter=1000)

start_pca = datetime.datetime.now()
logisticRegr_pca.fit(X_train_pca, y_train)
end_pca = datetime.datetime.now()

print("\nTraining time with PCA:", end_pca - start_pca)

acc_train_pca = accuracy_score(y_train, logisticRegr_pca.predict(X_train_pca))
acc_test_pca = accuracy_score(y_test, logisticRegr_pca.predict(X_test_pca))

print("Performance with PCA:\nTrain accuracy:", acc_train_pca, "\nTest accuracy:", acc_test_pca)


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training time without PCA: 0:05:22.306261
Performance without PCA:
Train accuracy: 0.8858928571428571 
Test accuracy: 0.8440714285714286


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


#Number of Components, Variance, Time Table


In [ ]:
pd.DataFrame(data = [[1.00, 784, 48.94, .9158],
                     [.99, 541, 34.69, .9169],
                     [.95, 330, 13.89, .92],
                     [.90, 236, 10.56, .9168],
                     [.85, 184, 8.85, .9156]],
             columns = ['Variance Retained',
                      'Number of Components',
                      'Time (seconds)',
                      'Accuracy'])

In [ ]:
#My own results

# pca(0.99)= n_componnets = mle (81), acc= 0.9167857142857143
# pca(0.85)= n_componnets = 150, acc= 0.916
# pca(0.7)= n_componnets = 53, acc= 0.906

